# Full-Duplex-Bench Vietnamese Evaluation Pipeline

Notebook này hướng dẫn chi tiết cách chạy thử nghiệm đánh giá hệ thống đàm thoại song song (Full-Duplex Spoken Dialogue) bằng tiếng Việt sử dụng hạ tầng GPU trên Kaggle hoặc Google Colab.

## 1. Clone Source Code từ GitHub

**Lưu ý:** Dự án này sử dụng branch `LamKD` từ repository `https://github.com/lamkdhe180931-arch/Full-Duplex-Bench.git`.

In [ ]:
# Di chuyển về thư mục gốc /kaggle/working trước khi clone để tránh bị đệ quy khi chạy lại cell nhiều lần
import os
import shutil
if os.path.exists('/kaggle/working'):
    %cd /kaggle/working
elif os.path.exists('/content'):
    %cd /content

# Xóa thư mục cũ nếu có để tránh clone lồng nhau
if os.path.exists('Full-Duplex-Bench'):
    shutil.rmtree('Full-Duplex-Bench')

!git clone -b LamKD https://github.com/lamkdhe180931-arch/Full-Duplex-Bench.git
%cd Full-Duplex-Bench/v1_v1.5

## 2. Cài đặt các thư viện hệ thống và thư viện Python

Bật GPU T4 trên Kaggle trước khi chạy cell này.

In [ ]:
# Cài đặt ffmpeg cho xử lý audio
!apt-get update && apt-get install -y ffmpeg

# Cài đặt các thư viện của Benchmark gốc và module sinh dữ liệu tự động
!pip install -r requirements.txt
!pip install -r data_generation/requirements.txt

## 3. Sinh dữ liệu Tiếng Việt Tự động (Dataset Generation)

Chạy các script tạo dữ liệu cho cả v1.0 và v1.5 từ các kịch bản JSON mẫu tiếng Việt đã chuẩn hóa.

In [ ]:
# Sinh dữ liệu v1.0 (Turn-Taking)
!python data_generation/v1_0/generate_v1_0.py

# Sinh dữ liệu v1.5 (Overlap)
!python data_generation/v1_5/generate_v1_5.py

## 4. Thiết lập khóa API Keys

Thay thế các giá trị bên dưới bằng API Key thực tế của bạn để tương tác với Gemini và GPT-4o phục vụ đánh giá hành vi.

In [ ]:
import os
os.environ["GEMINI_API_KEY"] = "YOUR_GEMINI_API_KEY"
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

## 5. Tương tác thử nghiệm với Agent (Model Inference)

Stream các tệp dữ liệu Tiếng Việt vừa tạo tới Agent và ghi âm lại các câu trả lời (`output.wav`). Ở đây ví dụ thử nghiệm với mô hình Gemini 2.5 Native Audio trên task `user_interruption` (v1.5).

In [ ]:
# ========================================== #
# 5.1. CHẠY INFERENCE CHO BENCHMARK V1.0
# ========================================== #
# Bạn có thể sử dụng model_inference/gemini/inference_gemini31_live.py (mô hình 3.1 Live) hoặc inference_gemini25_native.py
!python model_inference/gemini/inference_gemini31_live.py --base-dir dataset/v1_0 --task synthetic_pause_handling --overwrite
!python model_inference/gemini/inference_gemini31_live.py --base-dir dataset/v1_0 --task candor_turn_taking --overwrite
!python model_inference/gemini/inference_gemini31_live.py --base-dir dataset/v1_0 --task synthetic_user_interruption --overwrite

# ========================================== #
# 5.2. CHẠY INFERENCE CHO BENCHMARK V1.5
# ========================================== #
# Task: user_interruption
!python model_inference/gemini/inference_gemini31_live.py --base-dir dataset/v1_5 --task user_interruption --overwrite
!python model_inference/gemini/inference_gemini31_live.py --base-dir dataset/v1_5 --task user_interruption --prefix clean_ --overwrite

# Task: user_backchannel
!python model_inference/gemini/inference_gemini31_live.py --base-dir dataset/v1_5 --task user_backchannel --overwrite
!python model_inference/gemini/inference_gemini31_live.py --base-dir dataset/v1_5 --task user_backchannel --prefix clean_ --overwrite

# Task: talking_to_other
!python model_inference/gemini/inference_gemini31_live.py --base-dir dataset/v1_5 --task talking_to_other --overwrite
!python model_inference/gemini/inference_gemini31_live.py --base-dir dataset/v1_5 --task talking_to_other --prefix clean_ --overwrite

# Task: background_speech
!python model_inference/gemini/inference_gemini31_live.py --base-dir dataset/v1_5 --task background_speech --overwrite
!python model_inference/gemini/inference_gemini31_live.py --base-dir dataset/v1_5 --task background_speech --prefix clean_ --overwrite


## 6. Thực hiện bóc băng ASR (Yêu cầu GPU CUDA)

Chạy mô hình NeMo ASR lấy chi tiết word-level timestamps của Agent phục vụ các bước đo đạc độ trễ và phân tích hành vi.

In [ ]:
# ========================================== #
# 6.1. BÓC BĂNG ASR CHO BENCHMARK V1.0
# ========================================== #
!python get_transcript/asr.py --root_dir dataset/v1_0/synthetic_pause_handling --audio_name output.wav
!python get_transcript/asr.py --root_dir dataset/v1_0/candor_turn_taking --audio_name output.wav
!python get_transcript/asr.py --root_dir dataset/v1_0/synthetic_user_interruption --audio_name output.wav --task user_interruption

# ========================================== #
# 6.2. BÓC BĂNG ASR CHO BENCHMARK V1.5
# ========================================== #
# Bóc băng cho task: user_interruption
!python get_transcript/asr.py --root_dir dataset/v1_5/user_interruption --audio_name output.wav --task user_interruption
!python get_transcript/asr.py --root_dir dataset/v1_5/user_interruption --audio_name clean_input.wav
!python get_transcript/asr.py --root_dir dataset/v1_5/user_interruption --audio_name clean_output.wav
!python get_transcript/asr.py --root_dir dataset/v1_5/user_interruption --audio_name input.wav

# Bóc băng cho task: user_backchannel
!python get_transcript/asr.py --root_dir dataset/v1_5/user_backchannel --audio_name output.wav
!python get_transcript/asr.py --root_dir dataset/v1_5/user_backchannel --audio_name clean_input.wav
!python get_transcript/asr.py --root_dir dataset/v1_5/user_backchannel --audio_name clean_output.wav
!python get_transcript/asr.py --root_dir dataset/v1_5/user_backchannel --audio_name input.wav

# Bóc băng cho task: talking_to_other
!python get_transcript/asr.py --root_dir dataset/v1_5/talking_to_other --audio_name output.wav
!python get_transcript/asr.py --root_dir dataset/v1_5/talking_to_other --audio_name clean_input.wav
!python get_transcript/asr.py --root_dir dataset/v1_5/talking_to_other --audio_name clean_output.wav
!python get_transcript/asr.py --root_dir dataset/v1_5/talking_to_other --audio_name input.wav

# Bóc băng cho task: background_speech
!python get_transcript/asr.py --root_dir dataset/v1_5/background_speech --audio_name output.wav
!python get_transcript/asr.py --root_dir dataset/v1_5/background_speech --audio_name clean_input.wav
!python get_transcript/asr.py --root_dir dataset/v1_5/background_speech --audio_name clean_output.wav
!python get_transcript/asr.py --root_dir dataset/v1_5/background_speech --audio_name input.wav


## 7. Thực hiện đánh giá kết quả (Evaluation)

Chạy các kịch bản đánh giá để tổng hợp các chỉ số.

In [ ]:
# ========================================== #
# 7.1. ĐÁNH GIÁ KẾT QUẢ CHO BENCHMARK V1.0
# ========================================== #
%cd evaluation
print("=== 1. Đánh giá Pause Handling ===")
!python evaluate.py --task pause_handling --root_dir ../dataset/v1_0/synthetic_pause_handling

print("\n=== 2. Đánh giá Smooth Turn Taking ===")
!python evaluate.py --task smooth_turn_taking --root_dir ../dataset/v1_0/candor_turn_taking

print("\n=== 3. Đánh giá User Interruption (v1.0) ===")
!python evaluate.py --task user_interruption --root_dir ../dataset/v1_0/synthetic_user_interruption
%cd ..

# ========================================== #
# 7.2. ĐÁNH GIÁ KẾT QUẢ CHO BENCHMARK V1.5
# ========================================== #
%cd evaluation
print("=== 1. Đánh giá User Interruption (v1.5) ===")
!python evaluate.py --task behavior --root_dir ../dataset/v1_5/user_interruption
!python evaluate.py --task general_before_after --root_dir ../dataset/v1_5/user_interruption
!python get_timing.py --root_dir ../dataset/v1_5/user_interruption

print("\n=== 2. Đánh giá User Backchannel (v1.5) ===")
!python evaluate.py --task backchannel --root_dir ../dataset/v1_5/user_backchannel
!python evaluate.py --task general_before_after --root_dir ../dataset/v1_5/user_backchannel
!python get_timing.py --root_dir ../dataset/v1_5/user_backchannel

print("\n=== 3. Đánh giá Talking to Other (v1.5) ===")
!python evaluate.py --task behavior --root_dir ../dataset/v1_5/talking_to_other
!python evaluate.py --task general_before_after --root_dir ../dataset/v1_5/talking_to_other
!python get_timing.py --root_dir ../dataset/v1_5/talking_to_other

print("\n=== 4. Đánh giá Background Speech (v1.5) ===")
!python evaluate.py --task behavior --root_dir ../dataset/v1_5/background_speech
!python evaluate.py --task general_before_after --root_dir ../dataset/v1_5/background_speech
!python get_timing.py --root_dir ../dataset/v1_5/background_speech
%cd ..


In [ ]:
# ==================================================================
# 8. TRỰC QUAN HÓA CHI TIẾT TỪNG BƯỚC ĐẦU VÀO / ĐẦU RA VÀ NGHE FILE GHI ÂM
# ==================================================================
import os
import json
import IPython.display as ipd

def load_v1_5_templates():
    templates_data = {}
    base_gen_path = "/kaggle/working/Full-Duplex-Bench/v1_v1.5/data_generation/v1_5/templates"
    for filename in ["user_interruption.json", "user_backchannel.json", "talking_to_other.json", "background_speech.json"]:
        path = os.path.join(base_gen_path, filename)
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
                for item in data:
                    templates_data[item["id"]] = item
    return templates_data

def inspect_v1_5_processing_steps():
    base_dir = "/kaggle/working/Full-Duplex-Bench/v1_v1.5/dataset/v1_5"
    if not os.path.exists(base_dir):
        print(f"Thư mục {base_dir} không tồn tại.")
        return
        
    templates = load_v1_5_templates()
    tasks = sorted(os.listdir(base_dir))
    
    for task in tasks:
        if task.startswith(".") or not os.path.isdir(os.path.join(base_dir, task)):
            continue
            
        task_path = os.path.join(base_dir, task)
        print(f"\n========================================================")
        print(f"📌 NHÓM TASK: {task.upper()}")
        print(f"========================================================")
        
        samples = sorted(os.listdir(task_path))
        for sample in samples:
            if sample.startswith(".") or not os.path.isdir(os.path.join(task_path, sample)):
                continue
            
            sample_path = os.path.join(task_path, sample)
            print(f"\n👉 Mẫu thử: {sample}")
            
            # Đọc metadata
            meta_path = os.path.join(sample_path, "metadata.json")
            tpl = templates.get(sample, {})
            context_text = ""
            current_turn_text = ""
            
            if os.path.exists(meta_path):
                with open(meta_path, "r", encoding="utf-8") as f:
                    m_data = json.load(f)
                context_text = m_data.get("context_text", "")
                current_turn_text = m_data.get("current_turn_text", "")
            
            # --- 1. HIỂN THỊ THÔNG TIN KỊCH BẢN ---
            print("\n📝 [Kịch bản Gốc]:")
            if context_text:
                print(f"   - Trợ lý AI đang nói: \"{context_text}\"")
            print(f"   - Người dùng nói chen: \"{current_turn_text}\"")
            if "overlap_text" in tpl and tpl["overlap_text"]:
                print(f"   - Nhiễu đè nền (nếu có): \"{tpl['overlap_text']}\"")
            
            # --- 2. KIỂM TRA TỪNG BƯỚC FILE ÂM THANH & ASR ---
            steps = [
                ("A. Đầu vào sạch (Clean User Input)", "clean_input.wav", "clean_input.json"),
                ("B. Đầu vào trộn/chen (Noisy User Input)", "input.wav", "input.json"),
                ("C. AI phản hồi sạch (Clean AI Response)", "clean_output.wav", "clean_output.json"),
                ("D. AI phản hồi có chen (Noisy AI Response)", "output.wav", "output.json")
            ]
            
            for step_title, wav_name, json_name in steps:
                wav_path = os.path.join(sample_path, wav_name)
                json_path = os.path.join(sample_path, json_name)
                
                print(f"\n--- {step_title} ---")
                
                # Check Audio
                if os.path.exists(wav_path):
                    print(f"   🔊 File âm thanh: {wav_name} (OK)")
                    ipd.display(ipd.Audio(wav_path))
                else:
                    print(f"   🔴 [THIẾU/CHƯA XỬ LÝ] File âm thanh: {wav_name}")
                    
                # Check ASR Transcription
                if os.path.exists(json_path):
                    with open(json_path, "r", encoding="utf-8") as f:
                        asr_data = json.load(f)
                    print(f"   🔤 Nhận dạng ASR: \"{asr_data.get('text', '')}\"")
                else:
                    print(f"   🔴 [THIẾU/CHƯA XỬ LÝ] Kết quả nhận dạng ASR: {json_name}")
            
            # --- 3. ĐỌC KẾT QUẢ ĐÁNH GIÁ (NẾU CÓ) ---
            print("\n📈 [Kết quả phân tích hành vi & thời gian]:")
            
            # Đọc đánh giá hành vi OpenAI
            tag_path = os.path.join(sample_path, "content_tag.json")
            if os.path.exists(tag_path):
                with open(tag_path, "r", encoding="utf-8") as f:
                    tag_data = json.load(f)
                print(f"   - Phân tích hành vi (gpt-4o): {tag_data.get('behaviour', [])}")
                print(f"   - Lý giải: {tag_data.get('reason', '')}")
            else:
                print("   - Phân tích hành vi (gpt-4o): 🔴 [CHƯA CÓ / CHƯA CHẠY EVAL]")
                
            # Đọc độ trễ timing
            latency_path = os.path.join(sample_path, "latency_intervals.json")
            if os.path.exists(latency_path):
                with open(latency_path, "r", encoding="utf-8") as f:
                    lat_data = json.load(f)
                print(f"   - Độ trễ dừng (Stop Latency): {lat_data.get('latency_stop_list', [])}")
                print(f"   - Độ trễ phản hồi (Response Latency): {lat_data.get('latency_resp_list', [])}")
            else:
                print("   - Chỉ số độ trễ thời gian: 🔴 [CHƯA CÓ / CHƯA CHẠY TIMING]")
                
            print("-" * 60)

inspect_v1_5_processing_steps()
